In [2]:

from pathlib import Path
import pandas as pd
import joblib

# Find the project root reliably
current = Path.cwd()
BASE_DIR = next(
    (p for p in [current, *current.parents]
     if (p / "models" / "carbon_model.pkl").exists()
     and (p / "data" / "processed").exists()),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError(
        "Project root not found. Check the notebook location."
    )

DATA_DIR = BASE_DIR / "data" / "processed"

# Load datasets
carbon_df = pd.read_csv(DATA_DIR / "carbon_intensity.csv")
region_df = pd.read_csv(DATA_DIR / "region_config.csv")
workload_df = pd.read_csv(DATA_DIR / "workload.csv")

# Load trained ML model
model = joblib.load(BASE_DIR / "models" / "carbon_model.pkl")

print("Project root:", BASE_DIR)
print("\nDataset shapes:")
print("Carbon:", carbon_df.shape)
print("Regions:", region_df.shape)
print("Workloads:", workload_df.shape)

print("\nCarbon columns:", carbon_df.columns.tolist())
print("Region columns:", region_df.columns.tolist())
print("Workload columns:", workload_df.columns.tolist())

print("\nModel features:", model.feature_names_in_)


Project root: c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler

Dataset shapes:
Carbon: (38976, 8)
Regions: (1160, 14)
Workloads: (7010740, 5)

Carbon columns: ['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage', 'hour', 'day_of_week', 'month']
Region columns: ['datacenter_id', 'provider', 'region', 'host_id', 'pes', 'mips_per_pe', 'ram_mb', 'storage_mb', 'bandwidth_mbps', 'idle_power_w', 'max_power_w', 'pue', 'cost_per_hour', 'total_cpu_mips']
Workload columns: ['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']

Model features: ['renewable_percentage' 'hour' 'day_of_week' 'month' 'carbon_lag_1'
 'carbon_lag_2' 'carbon_lag_3']


In [3]:

print("WORKLOAD SUMMARY")
display(workload_df.head())
display(workload_df[["CPU", "Memory"]].describe())
print(workload_df.isnull().sum())

print("\nCLOUD RESOURCE SUMMARY")
display(region_df.head())
print("Unique providers:", region_df["provider"].nunique())
print("Unique regions:", region_df["region"].nunique())
print("Unique hosts:", region_df["host_id"].nunique())

display(
    region_df[
        [
            "pes",
            "total_cpu_mips",
            "ram_mb",
            "idle_power_w",
            "max_power_w",
            "pue",
            "cost_per_hour",
        ]
    ].describe()
)

carbon_df["timestamp"] = pd.to_datetime(
    carbon_df["timestamp"], utc=True
)

print("\nCARBON DATA SUMMARY")
print("Time range:", carbon_df["timestamp"].min(), "to", carbon_df["timestamp"].max())

carbon_regions = set(zip(carbon_df["provider"], carbon_df["region"]))
resource_regions = set(zip(region_df["provider"], region_df["region"]))

print("Regions with carbon history:", len(carbon_regions))
print("Configured regions:", len(resource_regions))
print("Configured regions without carbon history:", len(resource_regions - carbon_regions))


WORKLOAD SUMMARY


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


,CPU,Memory
count,7.010740e+06,7.010740e+06
mean,6.792311e+00,1.723452e-03
std,1.872698e+02,3.408144e-03
min,9.536743e-07,9.536743e-07
25%,2.257323e-02,3.786087e-04
50%,1.098804e-01,9.994507e-04
75%,7.098033e-01,1.642227e-03
max,2.399543e+04,4.711914e-01


Job_ID          0
Task_ID         0
Arrival_Time    0
CPU             0
Memory          0
dtype: int64

CLOUD RESOURCE SUMMARY


,datacenter_id,provider,region,host_id,pes,mips_per_pe,ram_mb,storage_mb,bandwidth_mbps,idle_power_w,max_power_w,pue,cost_per_hour,total_cpu_mips
0,DC001,aws,af-south-1,H01,4,800,8192,500000,5000,130,262,1.2,0.085,3200
1,DC001,aws,af-south-1,H02,8,1000,16384,1000000,10000,160,324,1.2,0.090,8000
2,DC001,aws,af-south-1,H03,8,1200,16384,1000000,10000,162,326,1.2,0.095,9600
3,DC001,aws,af-south-1,H04,12,1500,24576,1500000,15000,192,388,1.2,0.100,18000
4,DC001,aws,af-south-1,H05,16,2000,32768,2000000,20000,222,450,1.2,0.105,32000


Unique providers: 6
Unique regions: 116
Unique hosts: 10


,pes,total_cpu_mips,ram_mb,idle_power_w,max_power_w,pue,cost_per_hour
count,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000
mean,9.600000,14160.000000,19660.800000,178.200000,355.000000,1.174310,0.086405
std,4.080975,10122.387561,8357.836864,31.717933,64.160336,0.016783,0.010758
min,4.000000,3200.000000,8192.000000,130.000000,262.000000,1.150000,0.055300
25%,8.000000,8000.000000,16384.000000,160.000000,324.000000,1.160000,0.080800
50%,8.000000,9600.000000,16384.000000,171.000000,335.000000,1.180000,0.085500
75%,12.000000,18000.000000,24576.000000,202.000000,398.000000,1.180000,0.095000
max,16.000000,32000.000000,32768.000000,232.000000,460.000000,1.200000,0.105000



CARBON DATA SUMMARY
Time range: 2026-08-14 04:05:40.897319+00:00 to 2026-09-22 05:43:52.560911+00:00
Regions with carbon history: 116
Configured regions: 116
Configured regions without carbon history: 0


In [4]:

sample_size = 1000

workload_sample = workload_df.head(sample_size).copy()

carbon_latest = (
    carbon_df.sort_values("timestamp")
    .groupby(["provider", "region"], as_index=False)
    .tail(1)
    .copy()
)

region_carbon = region_df.merge(
    carbon_latest[
        [
            "provider",
            "region",
            "timestamp",
            "carbon_intensity_gco2_kwh",
            "renewable_percentage",
            "hour",
            "day_of_week",
            "month",
        ]
    ],
    on=["provider", "region"],
    how="left",
    validate="many_to_one",
)

print("Sample workload:", workload_sample.shape)
print("Resource-carbon mapping:", region_carbon.shape)
print("Missing carbon values:", region_carbon["carbon_intensity_gco2_kwh"].isna().sum())

display(
    region_carbon[
        [
            "provider",
            "region",
            "host_id",
            "total_cpu_mips",
            "ram_mb",
            "carbon_intensity_gco2_kwh",
            "renewable_percentage",
            "cost_per_hour",
        ]
    ].head(10)
)


Sample workload: (1000, 5)
Resource-carbon mapping: (1160, 20)
Missing carbon values: 0


,provider,region,host_id,total_cpu_mips,ram_mb,carbon_intensity_gco2_kwh,renewable_percentage,cost_per_hour
0,aws,af-south-1,H01,3200,8192,780.0,8.0,0.085
1,aws,af-south-1,H02,8000,16384,780.0,8.0,0.090
2,aws,af-south-1,H03,9600,16384,780.0,8.0,0.095
3,aws,af-south-1,H04,18000,24576,780.0,8.0,0.100
4,aws,af-south-1,H05,32000,32768,780.0,8.0,0.105
5,aws,af-south-1,H06,3200,8192,780.0,8.0,0.085
6,aws,af-south-1,H07,8000,16384,780.0,8.0,0.090
7,aws,af-south-1,H08,9600,16384,780.0,8.0,0.095
8,aws,af-south-1,H09,18000,24576,780.0,8.0,0.100
9,aws,af-south-1,H10,32000,32768,780.0,8.0,0.105


In [5]:

carbon_history = carbon_df.sort_values(
    ["provider", "region", "timestamp"]
).copy()

grouped = carbon_history.groupby(["provider", "region"])[
    "carbon_intensity_gco2_kwh"
]

carbon_history["carbon_lag_1"] = grouped.shift(1)
carbon_history["carbon_lag_2"] = grouped.shift(2)
carbon_history["carbon_lag_3"] = grouped.shift(3)

latest_features = (
    carbon_history.groupby(["provider", "region"], group_keys=False)
    .tail(1)
    .copy()
)

latest_features = latest_features.dropna(
    subset=[
        "renewable_percentage",
        "hour",
        "day_of_week",
        "month",
        "carbon_lag_1",
        "carbon_lag_2",
        "carbon_lag_3",
    ]
)

feature_columns = list(model.feature_names_in_)
X = latest_features[feature_columns]

latest_features["predicted_carbon_intensity"] = model.predict(X)

print("Regions predicted:", len(latest_features))
display(
    latest_features[
        [
            "provider",
            "region",
            "carbon_intensity_gco2_kwh",
            "predicted_carbon_intensity",
            "renewable_percentage",
        ]
    ].head(10)
)


Regions predicted: 116


,provider,region,carbon_intensity_gco2_kwh,predicted_carbon_intensity,renewable_percentage
335,aws,af-south-1,780.0,780.035065,8.0
671,aws,ap-east-1,281.5,284.309519,29.4
1007,aws,ap-northeast-1,308.3,310.088828,17.9
1343,aws,ap-northeast-2,280.5,282.899278,29.8
1679,aws,ap-northeast-3,285.0,286.971006,27.8
2015,aws,ap-south-1,412.5,412.500000,37.5
2351,aws,ap-south-2,375.0,374.933549,45.0
2687,aws,ap-southeast-1,298.5,306.807007,22.1
3023,aws,ap-southeast-2,433.2,368.581913,49.1
3359,aws,ca-central-1,30.0,29.925137,95.0


In [6]:

predicted_regions = latest_features[
    [
        "provider",
        "region",
        "predicted_carbon_intensity",
        "renewable_percentage",
    ]
].copy()

scheduler_resources = region_df.merge(
    predicted_regions,
    on=["provider", "region"],
    how="left",
    validate="many_to_one",
)

scheduler_resources = scheduler_resources.sort_values(
    "predicted_carbon_intensity"
).reset_index(drop=True)

print("Total resource records:", len(scheduler_resources))
print(
    "Missing predictions:",
    scheduler_resources["predicted_carbon_intensity"].isna().sum(),
)

display(
    scheduler_resources[
        [
            "provider",
            "region",
            "host_id",
            "total_cpu_mips",
            "ram_mb",
            "predicted_carbon_intensity",
            "renewable_percentage",
            "cost_per_hour",
        ]
    ].head(10)
)


Total resource records: 1160
Missing predictions: 0


,provider,region,host_id,total_cpu_mips,ram_mb,predicted_carbon_intensity,renewable_percentage,cost_per_hour
0,gcp,europe-west6,H07,8000,16384,24.137879,100.0,0.0810
1,gcp,europe-west6,H08,9600,16384,24.137879,100.0,0.0855
2,gcp,europe-west6,H10,32000,32768,24.137879,100.0,0.0945
3,gcp,europe-west6,H09,18000,24576,24.137879,100.0,0.0900
4,gcp,europe-west6,H05,32000,32768,24.137879,100.0,0.0945
5,gcp,europe-west6,H06,3200,8192,24.137879,100.0,0.0765
6,gcp,europe-west6,H04,18000,24576,24.137879,100.0,0.0900
7,gcp,europe-west6,H03,9600,16384,24.137879,100.0,0.0855
8,azure,switzerlandnorth,H07,8000,16384,24.137879,100.0,0.0855
9,azure,switzerlandnorth,H08,9600,16384,24.137879,100.0,0.0902


In [7]:

print("WORKLOAD REQUIREMENTS")
display(workload_df[["CPU", "Memory"]].describe())
display(workload_df[["CPU", "Memory"]].head(10))

print("\nRESOURCE CAPACITY")
display(
    region_df[
        [
            "pes",
            "mips_per_pe",
            "total_cpu_mips",
            "ram_mb",
            "cost_per_hour",
            "idle_power_w",
            "max_power_w",
        ]
    ].describe()
)

print("\nSAMPLE RESOURCE RECORDS")
display(
    region_df[
        [
            "provider",
            "region",
            "host_id",
            "pes",
            "mips_per_pe",
            "total_cpu_mips",
            "ram_mb",
        ]
    ].head(10)
)


WORKLOAD REQUIREMENTS


,CPU,Memory
count,7.010740e+06,7.010740e+06
mean,6.792311e+00,1.723452e-03
std,1.872698e+02,3.408144e-03
min,9.536743e-07,9.536743e-07
25%,2.257323e-02,3.786087e-04
50%,1.098804e-01,9.994507e-04
75%,7.098033e-01,1.642227e-03
max,2.399543e+04,4.711914e-01


,CPU,Memory
0,0.017989,0.000057
1,0.008105,0.000057
2,0.112489,0.000237
3,0.055855,0.001207
4,0.323642,0.003185
5,0.289879,0.003292
6,0.324837,0.003220
7,0.275387,0.003002
8,0.342675,0.003368
9,0.321858,0.003277



RESOURCE CAPACITY


,pes,mips_per_pe,total_cpu_mips,ram_mb,cost_per_hour,idle_power_w,max_power_w
count,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000
mean,9.600000,1300.000000,14160.000000,19660.800000,0.086405,178.200000,355.000000
std,4.080975,419.704485,10122.387561,8357.836864,0.010758,31.717933,64.160336
min,4.000000,800.000000,3200.000000,8192.000000,0.055300,130.000000,262.000000
25%,8.000000,1000.000000,8000.000000,16384.000000,0.080800,160.000000,324.000000
50%,8.000000,1200.000000,9600.000000,16384.000000,0.085500,171.000000,335.000000
75%,12.000000,1500.000000,18000.000000,24576.000000,0.095000,202.000000,398.000000
max,16.000000,2000.000000,32000.000000,32768.000000,0.105000,232.000000,460.000000



SAMPLE RESOURCE RECORDS


,provider,region,host_id,pes,mips_per_pe,total_cpu_mips,ram_mb
0,aws,af-south-1,H01,4,800,3200,8192
1,aws,af-south-1,H02,8,1000,8000,16384
2,aws,af-south-1,H03,8,1200,9600,16384
3,aws,af-south-1,H04,12,1500,18000,24576
4,aws,af-south-1,H05,16,2000,32000,32768
5,aws,af-south-1,H06,4,800,3200,8192
6,aws,af-south-1,H07,8,1000,8000,16384
7,aws,af-south-1,H08,8,1200,9600,16384
8,aws,af-south-1,H09,12,1500,18000,24576
9,aws,af-south-1,H10,16,2000,32000,32768


In [8]:

workload_sample = workload_df.head(1000).copy()

print("CPU values:", workload_sample["CPU"].describe())
print("Memory values:", workload_sample["Memory"].describe())
print("Host CPU capacity:", region_df["total_cpu_mips"].describe())
print("Host memory capacity:", region_df["ram_mb"].describe())

print("\nFirst 10 tasks:")
display(workload_sample.head(10))

print("\nFirst 10 hosts:")
display(
    region_df[
        [
            "provider",
            "region",
            "host_id",
            "pes",
            "mips_per_pe",
            "total_cpu_mips",
            "ram_mb",
            "cost_per_hour",
        ]
    ].head(10)
)


CPU values: count    1000.000000
mean        1.882433
std         5.009051
min         0.000576
25%         0.025284
50%         0.127747
75%         0.338079
max        43.530272
Name: CPU, dtype: float64
Memory values: count    1000.000000
mean        0.001885
std         0.002573
min         0.000005
25%         0.000614
50%         0.001146
75%         0.002576
max         0.031372
Name: Memory, dtype: float64
Host CPU capacity: count     1160.000000
mean     14160.000000
std      10122.387561
min       3200.000000
25%       8000.000000
50%       9600.000000
75%      18000.000000
max      32000.000000
Name: total_cpu_mips, dtype: float64
Host memory capacity: count     1160.000000
mean     19660.800000
std       8357.836864
min       8192.000000
25%      16384.000000
50%      16384.000000
75%      24576.000000
max      32768.000000
Name: ram_mb, dtype: float64

First 10 tasks:


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185
5,736959135802,0,609764.644,0.289879,0.003292
6,736959135804,0,609781.475,0.324837,0.003220
7,736959135806,0,609781.502,0.275387,0.003002
8,736959135808,0,609781.529,0.342675,0.003368
9,736959135810,0,609781.556,0.321858,0.003277



First 10 hosts:


,provider,region,host_id,pes,mips_per_pe,total_cpu_mips,ram_mb,cost_per_hour
0,aws,af-south-1,H01,4,800,3200,8192,0.085
1,aws,af-south-1,H02,8,1000,8000,16384,0.090
2,aws,af-south-1,H03,8,1200,9600,16384,0.095
3,aws,af-south-1,H04,12,1500,18000,24576,0.100
4,aws,af-south-1,H05,16,2000,32000,32768,0.105
5,aws,af-south-1,H06,4,800,3200,8192,0.085
6,aws,af-south-1,H07,8,1000,8000,16384,0.090
7,aws,af-south-1,H08,8,1200,9600,16384,0.095
8,aws,af-south-1,H09,12,1500,18000,24576,0.100
9,aws,af-south-1,H10,16,2000,32000,32768,0.105


In [9]:

print("WORKLOAD SAMPLE")
display(workload_df[["Job_ID", "Task_ID", "Arrival_Time", "CPU", "Memory"]].head(15))

print("\nCPU AND MEMORY QUANTILES")
display(workload_df[["CPU", "Memory"]].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1]))

print("\nRESOURCE CAPACITY SAMPLE")
display(
    region_df[
        ["provider", "region", "host_id", "pes", "mips_per_pe", "total_cpu_mips", "ram_mb"]
    ].head(10)
)

print("\nWORKLOAD DATA TYPES")
print(workload_df.dtypes)

print("\nRESOURCE DATA TYPES")
print(region_df[["pes", "mips_per_pe", "total_cpu_mips", "ram_mb"]].dtypes)


WORKLOAD SAMPLE


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185
5,736959135802,0,609764.644,0.289879,0.003292
6,736959135804,0,609781.475,0.324837,0.003220
7,736959135806,0,609781.502,0.275387,0.003002
8,736959135808,0,609781.529,0.342675,0.003368
9,736959135810,0,609781.556,0.321858,0.003277



CPU AND MEMORY QUANTILES


,CPU,Memory
0.00,9.536743e-07,9.536743e-07
0.25,2.257323e-02,3.786087e-04
0.50,1.098804e-01,9.994507e-04
0.75,7.098033e-01,1.642227e-03
0.90,1.064812e+01,4.875183e-03
0.99,3.922404e+01,7.888794e-03
1.00,2.399543e+04,4.711914e-01



RESOURCE CAPACITY SAMPLE


,provider,region,host_id,pes,mips_per_pe,total_cpu_mips,ram_mb
0,aws,af-south-1,H01,4,800,3200,8192
1,aws,af-south-1,H02,8,1000,8000,16384
2,aws,af-south-1,H03,8,1200,9600,16384
3,aws,af-south-1,H04,12,1500,18000,24576
4,aws,af-south-1,H05,16,2000,32000,32768
5,aws,af-south-1,H06,4,800,3200,8192
6,aws,af-south-1,H07,8,1000,8000,16384
7,aws,af-south-1,H08,8,1200,9600,16384
8,aws,af-south-1,H09,12,1500,18000,24576
9,aws,af-south-1,H10,16,2000,32000,32768



WORKLOAD DATA TYPES
Job_ID            int64
Task_ID           int64
Arrival_Time    float64
CPU             float64
Memory          float64
dtype: object

RESOURCE DATA TYPES
pes               int64
mips_per_pe       int64
total_cpu_mips    int64
ram_mb            int64
dtype: object


In [10]:

print(workload_df[["CPU", "Memory"]].describe().to_string())

print("\nCPU unique values:")
print(workload_df["CPU"].head(20).tolist())

print("\nMemory unique values:")
print(workload_df["Memory"].head(20).tolist())

print("\nArrival time range:")
print(workload_df["Arrival_Time"].min(), workload_df["Arrival_Time"].max())

print("\nWorkload source files:")
print(list((BASE_DIR / "data").rglob("*workload*")))


                CPU        Memory
count  7.010740e+06  7.010740e+06
mean   6.792311e+00  1.723452e-03
std    1.872698e+02  3.408144e-03
min    9.536743e-07  9.536743e-07
25%    2.257323e-02  3.786087e-04
50%    1.098804e-01  9.994507e-04
75%    7.098033e-01  1.642227e-03
max    2.399543e+04  4.711914e-01

CPU unique values:
[0.017989158630371, 0.0081052780151367, 0.1124887466430664, 0.0558547973632812, 0.3236417770385742, 0.2898788452148437, 0.3248367309570312, 0.2753868103027344, 0.3426752090454101, 0.3218584060668945, 0.3398427963256836, 0.300088882446289, 0.3270320892333984, 0.3770551681518554, 0.3181114196777344, 0.334721565246582, 0.2954177856445312, 0.0080852508544921, 0.018899917602539, 0.0123853683471679]

Memory unique values:
[5.7220458984375e-05, 5.7220458984375e-05, 0.0002374649047851, 0.0012073516845703, 0.0031852722167968, 0.0032920837402343, 0.0032196044921875, 0.0030021667480468, 0.0033683776855468, 0.0032768249511718, 0.0032310485839843, 0.0033454895019531, 0.003364562

In [11]:

preprocessed_path = BASE_DIR / "data" / "preprocessed" / "workload.csv"
processed_path = BASE_DIR / "data" / "processed" / "workload.csv"

preprocessed_df = pd.read_csv(preprocessed_path)
processed_df = pd.read_csv(processed_path)

print("Preprocessed shape:", preprocessed_df.shape)
print("Processed shape:", processed_df.shape)

print("\nPreprocessed columns:")
print(preprocessed_df.columns.tolist())

print("\nProcessed columns:")
print(processed_df.columns.tolist())

print("\nPreprocessed sample:")
display(preprocessed_df.head())

print("\nProcessed sample:")
display(processed_df.head())

print("\nPreprocessed CPU and Memory:")
display(preprocessed_df[["CPU", "Memory"]].describe())

print("\nProcessed CPU and Memory:")
display(processed_df[["CPU", "Memory"]].describe())


Preprocessed shape: (7200798, 5)
Processed shape: (7010740, 5)

Preprocessed columns:
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']

Processed columns:
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']

Preprocessed sample:


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185



Processed sample:


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185



Preprocessed CPU and Memory:


,CPU,Memory
count,7.200798e+06,7.200798e+06
mean,6.615248e+00,1.679562e-03
std,1.848327e+02,3.373489e-03
min,0.000000e+00,0.000000e+00
25%,1.876068e-02,3.480911e-04
50%,1.038666e-01,9.727478e-04
75%,6.178150e-01,1.592636e-03
max,2.399543e+04,4.711914e-01



Processed CPU and Memory:


,CPU,Memory
count,7.010740e+06,7.010740e+06
mean,6.792311e+00,1.723452e-03
std,1.872698e+02,3.408144e-03
min,9.536743e-07,9.536743e-07
25%,2.257323e-02,3.786087e-04
50%,1.098804e-01,9.994507e-04
75%,7.098033e-01,1.642227e-03
max,2.399543e+04,4.711914e-01


In [12]:

print("PREPROCESSED DATA")
display(preprocessed_df.head(10))
display(preprocessed_df[["CPU", "Memory"]].describe())

print("\nPROCESSED DATA")
display(processed_df.head(10))
display(processed_df[["CPU", "Memory"]].describe())

print("\nMISSING VALUES")
print("Preprocessed:")
print(preprocessed_df.isnull().sum())
print("\nProcessed:")
print(processed_df.isnull().sum())

print("\nDUPLICATE TASK IDs")
print("Preprocessed:", preprocessed_df.duplicated(["Job_ID", "Task_ID"]).sum())
print("Processed:", processed_df.duplicated(["Job_ID", "Task_ID"]).sum())

print("\nPREPROCESSING CODE")
preprocessing_path = BASE_DIR / "src" / "cloudsim" / "preprocessing.py"
print(preprocessing_path.read_text(encoding="utf-8"))


PREPROCESSED DATA


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185
5,736959135802,0,609764.644,0.289879,0.003292
6,736959135804,0,609781.475,0.324837,0.003220
7,736959135806,0,609781.502,0.275387,0.003002
8,736959135808,0,609781.529,0.342675,0.003368
9,736959135810,0,609781.556,0.321858,0.003277


,CPU,Memory
count,7.200798e+06,7.200798e+06
mean,6.615248e+00,1.679562e-03
std,1.848327e+02,3.373489e-03
min,0.000000e+00,0.000000e+00
25%,1.876068e-02,3.480911e-04
50%,1.038666e-01,9.727478e-04
75%,6.178150e-01,1.592636e-03
max,2.399543e+04,4.711914e-01



PROCESSED DATA


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185
5,736959135802,0,609764.644,0.289879,0.003292
6,736959135804,0,609781.475,0.324837,0.003220
7,736959135806,0,609781.502,0.275387,0.003002
8,736959135808,0,609781.529,0.342675,0.003368
9,736959135810,0,609781.556,0.321858,0.003277


,CPU,Memory
count,7.010740e+06,7.010740e+06
mean,6.792311e+00,1.723452e-03
std,1.872698e+02,3.408144e-03
min,9.536743e-07,9.536743e-07
25%,2.257323e-02,3.786087e-04
50%,1.098804e-01,9.994507e-04
75%,7.098033e-01,1.642227e-03
max,2.399543e+04,4.711914e-01



MISSING VALUES
Preprocessed:
Job_ID          0
Task_ID         0
Arrival_Time    0
CPU             0
Memory          0
dtype: int64

Processed:
Job_ID          0
Task_ID         0
Arrival_Time    0
CPU             0
Memory          0
dtype: int64

DUPLICATE TASK IDs
Preprocessed: 0
Processed: 0

PREPROCESSING CODE
from pathlib import Path
import pandas as pd

def load_carbon_data(file_path):
    df = pd.read_csv(file_path)

    required_columns = [
        "provider",
        "region",
        "timestamp",
        "carbon_intensity_gco2_kwh",
        "renewable_percentage",
    ]

    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Missing carbon columns: {missing}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["carbon_intensity_gco2_kwh"] = pd.to_numeric(
        df["carbon_intensity_gco2_kwh"], errors="coerce"
    )
    df["renewable_percentage"] = pd.to_numeric(
        df["renewable_percentage"], errors

In [13]:

preprocessing_path = BASE_DIR / "src" / "cloudsim" / "preprocessing.py"
source = preprocessing_path.read_text(encoding="utf-8")
print(source)


from pathlib import Path
import pandas as pd

def load_carbon_data(file_path):
    df = pd.read_csv(file_path)

    required_columns = [
        "provider",
        "region",
        "timestamp",
        "carbon_intensity_gco2_kwh",
        "renewable_percentage",
    ]

    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Missing carbon columns: {missing}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["carbon_intensity_gco2_kwh"] = pd.to_numeric(
        df["carbon_intensity_gco2_kwh"], errors="coerce"
    )
    df["renewable_percentage"] = pd.to_numeric(
        df["renewable_percentage"], errors="coerce"
    )

    df = df.dropna(subset=required_columns)
    df = df.drop_duplicates()
    df = df.sort_values(
        ["provider", "region", "timestamp"]
    ).reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"]

In [14]:

import ast

source = (BASE_DIR / "src" / "cloudsim" / "preprocessing.py").read_text(encoding="utf-8")
tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        if "workload" in node.name.lower() or "task" in node.name.lower():
            print(ast.get_source_segment(source, node))
            print()


def load_workload_data(file_path, chunksize=250_000):
    required_columns = [
        "Job_ID",
        "Task_ID",
        "Arrival_Time",
        "CPU",
        "Memory",
    ]

    chunks = []

    for chunk in pd.read_csv(file_path, chunksize=chunksize):
        missing = set(required_columns) - set(chunk.columns)
        if missing:
            raise ValueError(f"Missing workload columns: {missing}")

        for column in ["Arrival_Time", "CPU", "Memory"]:
            chunk[column] = pd.to_numeric(
                chunk[column], errors="coerce"
            )

        chunk = chunk.dropna(subset=required_columns)

        chunk = chunk[
            (chunk["Arrival_Time"] >= 0)
            & (chunk["CPU"] > 0)
            & (chunk["Memory"] > 0)
        ]

        chunks.append(chunk)

    if not chunks:
        return pd.DataFrame(columns=required_columns)

    return pd.concat(chunks, ignore_index=True)



In [15]:

candidate_resources = scheduler_resources[
    [
        "provider",
        "region",
        "host_id",
        "total_cpu_mips",
        "ram_mb",
        "predicted_carbon_intensity",
        "renewable_percentage",
        "cost_per_hour",
    ]
].copy()

candidate_resources = candidate_resources.dropna(
    subset=[
        "total_cpu_mips",
        "ram_mb",
        "predicted_carbon_intensity",
        "cost_per_hour",
    ]
)

candidate_resources = candidate_resources[
    (candidate_resources["total_cpu_mips"] > 0)
    & (candidate_resources["ram_mb"] > 0)
    & (candidate_resources["cost_per_hour"] >= 0)
]

candidate_resources = candidate_resources.sort_values(
    ["predicted_carbon_intensity", "cost_per_hour"],
    ascending=[True, True],
).reset_index(drop=True)

print("Eligible resource records:", len(candidate_resources))
display(candidate_resources.head(10))


Eligible resource records: 1160


,provider,region,host_id,total_cpu_mips,ram_mb,predicted_carbon_intensity,renewable_percentage,cost_per_hour
0,gcp,europe-west6,H06,3200,8192,24.137879,100.0,0.0765
1,gcp,europe-west6,H01,3200,8192,24.137879,100.0,0.0765
2,azure,switzerlandnorth,H06,3200,8192,24.137879,100.0,0.0808
3,azure,switzerlandnorth,H01,3200,8192,24.137879,100.0,0.0808
4,gcp,europe-west6,H07,8000,16384,24.137879,100.0,0.0810
5,gcp,europe-west6,H02,8000,16384,24.137879,100.0,0.0810
6,aws,eu-central-2,H06,3200,8192,24.137879,100.0,0.0850
7,aws,eu-central-2,H01,3200,8192,24.137879,100.0,0.0850
8,gcp,europe-west6,H08,9600,16384,24.137879,100.0,0.0855
9,gcp,europe-west6,H03,9600,16384,24.137879,100.0,0.0855


In [16]:

print(
    scheduler_resources["predicted_carbon_intensity"]
    .describe()
    .to_string()
)

print("\nUnique predicted values:")
print(
    scheduler_resources["predicted_carbon_intensity"].nunique()
)

print("\nLowest predicted carbon intensity:")
display(
    scheduler_resources[
        [
            "provider",
            "region",
            "predicted_carbon_intensity",
            "renewable_percentage",
        ]
    ]
    .drop_duplicates(["provider", "region"])
    .sort_values("predicted_carbon_intensity")
    .head(10)
)

print("\nHighest predicted carbon intensity:")
display(
    scheduler_resources[
        [
            "provider",
            "region",
            "predicted_carbon_intensity",
            "renewable_percentage",
        ]
    ]
    .drop_duplicates(["provider", "region"])
    .sort_values("predicted_carbon_intensity", ascending=False)
    .head(10)
)


count    1160.000000
mean      292.943284
std       173.015876
min        24.137879
25%       130.321715
50%       308.376765
75%       412.500000
max       780.035065

Unique predicted values:
40

Lowest predicted carbon intensity:


,provider,region,predicted_carbon_intensity,renewable_percentage
0,gcp,europe-west6,24.137879,100.0
8,azure,switzerlandnorth,24.137879,100.0
20,aws,eu-central-2,24.137879,100.0
30,azure,norwayeast,27.226575,98.9
40,azure,swedencentral,28.756832,33.9
44,aws,eu-north-1,28.756832,33.9
60,ovh,bhs,29.925137,95.0
64,aws,ca-central-1,29.925137,95.0
76,gcp,northamerica-northeast1,29.925137,95.0
86,azure,canadaeast,29.925137,95.0



Highest predicted carbon intensity:


,provider,region,predicted_carbon_intensity,renewable_percentage
1148,azure,southafricanorth,780.035065,8.0
1130,gcp,africa-south1,780.035065,8.0
1136,aws,af-south-1,780.035065,8.0
1120,azure,westus3,551.469636,10.5
1100,azure,australiasoutheast,541.331541,30.5
1108,gcp,australia-southeast2,541.331541,30.5
1071,azure,westeurope,531.897752,0.4
1070,gcp,europe-west4,531.897752,0.4
1075,scaleway,nl-ams,531.897752,0.4
1043,ovh,waw,525.839817,23.4


In [17]:

import ast

scheduler_source = (BASE_DIR / "src" / "cloudsim" / "scheduler.py").read_text(encoding="utf-8")
scheduler_tree = ast.parse(scheduler_source)

for node in scheduler_tree.body:
    if isinstance(node, ast.FunctionDef):
        print(ast.get_source_segment(scheduler_source, node))
        print()


def select_best_region(
    candidates,
    carbon_weight=0.6,
    energy_weight=0.25,
    cost_weight=0.15,
):
    required = {
        "provider",
        "region",
        "predicted_carbon",
        "estimated_energy_kwh",
        "estimated_cost",
    }

    missing = required - set(candidates.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    if candidates.empty:
        raise ValueError("No candidate regions available.")

    if min(carbon_weight, energy_weight, cost_weight) < 0:
        raise ValueError("Weights must be non-negative.")

    if carbon_weight + energy_weight + cost_weight == 0:
        raise ValueError("At least one weight must be positive.")

    candidates = candidates.copy()

    for column in [
        "predicted_carbon",
        "estimated_energy_kwh",
        "estimated_cost",
    ]:
        candidates[column] = pd.to_numeric(
            candidates[column], errors="coerce"
        )

    candidates = candidates.dropna(
  

In [18]:

lines = scheduler_source.splitlines()

for i, line in enumerate(lines):
    if any(term in line for term in [
        "estimated_energy_kwh",
        "estimated_cost",
        "total_weight",
        "score",
        "carbon_weight",
        "energy_weight",
        "cost_weight",
    ]):
        start = max(0, i - 2)
        end = min(len(lines), i + 3)
        print(f"\nLines {start + 1}-{end}")
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(start, end)
        ))



Lines 5-9
5: def select_best_region(
6:     candidates,
7:     carbon_weight=0.6,
8:     energy_weight=0.25,
9:     cost_weight=0.15,

Lines 6-10
6:     candidates,
7:     carbon_weight=0.6,
8:     energy_weight=0.25,
9:     cost_weight=0.15,
10: ):

Lines 7-11
7:     carbon_weight=0.6,
8:     energy_weight=0.25,
9:     cost_weight=0.15,
10: ):
11:     required = {

Lines 13-17
13:         "region",
14:         "predicted_carbon",
15:         "estimated_energy_kwh",
16:         "estimated_cost",
17:     }

Lines 14-18
14:         "predicted_carbon",
15:         "estimated_energy_kwh",
16:         "estimated_cost",
17:     }
18: 

Lines 24-28
24:         raise ValueError("No candidate regions available.")
25: 
26:     if min(carbon_weight, energy_weight, cost_weight) < 0:
27:         raise ValueError("Weights must be non-negative.")
28: 

Lines 27-31
27:         raise ValueError("Weights must be non-negative.")
28: 
29:     if carbon_weight + energy_weight + cost_weight == 0:
30:      

In [19]:

print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(35, min(86, len(lines)))
))


36:         "estimated_energy_kwh",
37:         "estimated_cost",
38:     ]:
39:         candidates[column] = pd.to_numeric(
40:             candidates[column], errors="coerce"
41:         )
42: 
43:     candidates = candidates.dropna(
44:         subset=[
45:             "predicted_carbon",
46:             "estimated_energy_kwh",
47:             "estimated_cost",
48:         ]
49:     )
50: 
51:     if candidates.empty:
52:         raise ValueError("No valid candidate estimates.")
53: 
54:     if (
55:         candidates[
56:             ["predicted_carbon", "estimated_energy_kwh", "estimated_cost"]
57:         ] < 0
58:     ).any().any():
59:         raise ValueError("Carbon, energy, and cost must be non-negative.")
60: 
61:     weights = [carbon_weight, energy_weight, cost_weight]
62:     columns = [
63:         "predicted_carbon",
64:         "estimated_energy_kwh",
65:         "estimated_cost",
66:     ]
67: 
68:     for column, weight in zip(columns, weights):
69:         minimum

In [20]:

print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(60, min(85, len(lines)))
))


61:     weights = [carbon_weight, energy_weight, cost_weight]
62:     columns = [
63:         "predicted_carbon",
64:         "estimated_energy_kwh",
65:         "estimated_cost",
66:     ]
67: 
68:     for column, weight in zip(columns, weights):
69:         minimum = candidates[column].min()
70:         maximum = candidates[column].max()
71: 
72:         if maximum > minimum:
73:             candidates[column + "_normalized"] = (
74:                 candidates[column] - minimum
75:             ) / (maximum - minimum)
76:         else:
77:             candidates[column + "_normalized"] = 0.0
78: 
79:     total_weight = sum(weights)
80: 
81:     candidates["score"] = sum(
82:         weight * candidates[column + "_normalized"]
83:         for column, weight in zip(columns, weights)
84:     ) / total_weight
85: 


In [21]:

from importlib.util import spec_from_file_location, module_from_spec

scheduler_file = BASE_DIR / "src" / "cloudsim" / "scheduler.py"
spec = spec_from_file_location("carbon_scheduler", scheduler_file)
scheduler_module = module_from_spec(spec)
spec.loader.exec_module(scheduler_module)

select_best_region = scheduler_module.select_best_region

execution_hours = 1.0

candidates = scheduler_resources.copy()

candidates["predicted_carbon"] = candidates[
    "predicted_carbon_intensity"
].clip(lower=0)

candidates["estimated_energy_kwh"] = (
    candidates["max_power_w"]
    * candidates["pue"]
    * execution_hours
    / 1000
)

candidates["estimated_cost"] = (
    candidates["cost_per_hour"] * execution_hours
)

ranked_candidates = select_best_region(
    candidates[
        [
            "provider",
            "region",
            "host_id",
            "predicted_carbon",
            "estimated_energy_kwh",
            "estimated_cost",
        ]
    ].copy()
)

display(ranked_candidates.head(10))


,provider,region,host_id,predicted_carbon,estimated_energy_kwh,estimated_cost,predicted_carbon_normalized,estimated_energy_kwh_normalized,estimated_cost_normalized,score
0,scaleway,fr-par,H01,46.235196,0.30130,0.0553,0.029233,0.000000,0.000000,0.017540
1,scaleway,fr-par,H06,46.235196,0.31280,0.0553,0.029233,0.045872,0.000000,0.029008
2,ovh,bhs,H01,29.925137,0.30392,0.0638,0.007656,0.010451,0.171026,0.032860
3,hetzner,hel1,H01,59.544060,0.30130,0.0595,0.046840,0.000000,0.084507,0.040780
4,ovh,bhs,H06,29.925137,0.31552,0.0638,0.007656,0.056721,0.171026,0.044428
5,ovh,rbx,H01,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
6,ovh,sbg,H01,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
7,ovh,gra,H01,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
8,hetzner,hel1,H06,59.544060,0.31280,0.0595,0.046840,0.045872,0.084507,0.052248
9,ovh,gra,H06,46.235196,0.31552,0.0638,0.029233,0.056721,0.171026,0.057374


In [22]:

results_dir = BASE_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

ranking_path = results_dir / "notebook4_ranked_candidates.csv"
ranked_candidates.to_csv(ranking_path, index=False)

print("Saved:", ranking_path)
print("Candidate rows:", len(ranked_candidates))
print("Top-ranked candidate:")
display(ranked_candidates.head(1))


Saved: c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\results\notebook4_ranked_candidates.csv
Candidate rows: 1160
Top-ranked candidate:


,provider,region,host_id,predicted_carbon,estimated_energy_kwh,estimated_cost,predicted_carbon_normalized,estimated_energy_kwh_normalized,estimated_cost_normalized,score
0,scaleway,fr-par,H01,46.235196,0.3013,0.0553,0.029233,0.0,0.0,0.01754


In [23]:

for folder in [BASE_DIR, BASE_DIR / "src", BASE_DIR / "src" / "cloudsim"]:
    print(f"\n{folder}")
    if folder.exists():
        for path in sorted(folder.iterdir()):
            print(path.name)

print("\nCloudSim and Maven files:")
for pattern in ["pom.xml", "*.java", "*cloudsim*", "*CloudSim*"]:
    matches = list(BASE_DIR.rglob(pattern))
    matches = [
        p for p in matches
        if not any(part in {"venv", "venv_windows", ".git", "__pycache__"} for part in p.parts)
    ]
    print(f"\n{pattern}:")
    for path in matches[:40]:
        print(path.relative_to(BASE_DIR))



c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler
.DS_Store
data
main.py
models
notebooks
requirements.txt
results
src
venv

c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\src
cloudsim

c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\src\cloudsim
__pycache__
carbon_prediction.py
data_inspection.py
evaluation.py
preprocessing.py
scheduler.py

CloudSim and Maven files:



pom.xml:

*.java:

*cloudsim*:
src\cloudsim

*CloudSim*:
src\cloudsim


In [24]:

for filename in ["main.py", "requirements.txt"]:
    path = BASE_DIR / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")
    print(path.read_text(encoding="utf-8"))

print("\nCloudSim-related files:")
for path in BASE_DIR.rglob("*"):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["cloudsim", "scheduler", "simulation"]
    ):
        if not any(
            part in {"venv", "venv_windows", ".git", "__pycache__"}
            for part in path.parts
        ):
            print(path.relative_to(BASE_DIR))



==================== main.py ====================


==================== requirements.txt ====================
anyio==4.15.1
appnope==1.0.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
arrow==1.4.0
asttokens==3.0.2
async-lru==2.3.0
attrs==26.1.0
babel==2.18.0
beautifulsoup4==4.15.0
bleach==6.4.0
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
cloudpickle==3.1.2
comm==0.2.3
contourpy==1.4.0
cycler==0.12.1
debugpy==1.8.22
defusedxml==0.7.1
executing==2.2.1
fastjsonschema==2.22.2
fonttools==4.65.0
fqdn==1.5.1
h11==0.16.0
httpcore==1.0.9
httpx==0.28.1
idna==3.20
ipykernel==7.3.0
ipython==9.17.1
ipython_pygments_lexers==1.1.1
ipywidgets==8.1.9
isoduration==20.11.0
jedi==0.20.0
Jinja2==3.1.6
joblib==1.6.0
json5==0.15.0
jsonpointer==3.1.1
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter==1.1.1
jupyter-console==6.6.3
jupyter-events==0.12.1
jupyter-lsp==2.3.1
jupyter_builder==1.2.3
jupyter_client==8.10.0
jupyter_core==5.9.1
jupyter_server==2.21.1
jupyter_server_termi

In [26]:
import shutil
import subprocess

java_path = shutil.which("java")
javac_path = shutil.which("javac")
maven_path = shutil.which("mvn")

print("Java:", java_path)
print("Javac:", javac_path)
print("Maven:", maven_path)

if java_path:
    result = subprocess.run(
        [java_path, "-version"],
        capture_output=True,
        text=True
    )
    print(result.stderr or result.stdout)

if maven_path:
    result = subprocess.run(
        ["cmd.exe", "/c", maven_path, "-version"],
        capture_output=True,
        text=True
    )
    print(result.stdout or result.stderr)

Java: C:\Program Files\Eclipse Adoptium\jdk-25.0.3.9-hotspot\bin\java.EXE
Javac: C:\Program Files\Eclipse Adoptium\jdk-25.0.3.9-hotspot\bin\javac.EXE
Maven: C:\Tools\apache-maven-3.10.0\bin\mvn.CMD
openjdk version "25.0.3" 2026-04-21 LTS
OpenJDK Runtime Environment Temurin-25.0.3+9 (build 25.0.3+9-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.3+9 (build 25.0.3+9-LTS, mixed mode, sharing)

Apache Maven 3.10.0 (c43a36b8d67be7e0805a411bc0898af1a51f5472)
Maven home: C:\Tools\apache-maven-3.10.0
Java version: 25.0.3, vendor: Eclipse Adoptium, runtime: C:\Program Files\Eclipse Adoptium\jdk-25.0.3.9-hotspot
Default locale: en_IN, platform encoding: UTF-8, time zone: Asia/Calcutta
OS name: "windows 11", version: "10.0", arch: "amd64", family: "windows"



In [27]:
scheduler_candidates = scheduler_resources.copy()

scheduler_candidates["predicted_carbon"] = (
    scheduler_candidates["predicted_carbon_intensity"]
)

execution_hours = 1.0

scheduler_candidates["estimated_energy_kwh"] = (
    scheduler_candidates["max_power_w"]
    * scheduler_candidates["pue"]
    * execution_hours / 1000
)

scheduler_candidates["estimated_cost"] = (
    scheduler_candidates["cost_per_hour"] * execution_hours
)

ranked_candidates = select_best_region(
    scheduler_candidates[
        [
            "provider",
            "region",
            "predicted_carbon",
            "estimated_energy_kwh",
            "estimated_cost"
        ]
    ]
)

print("Total regions:", len(ranked_candidates))
display(ranked_candidates.head(10))

Total regions: 1160


,provider,region,predicted_carbon,estimated_energy_kwh,estimated_cost,predicted_carbon_normalized,estimated_energy_kwh_normalized,estimated_cost_normalized,score
0,scaleway,fr-par,46.235196,0.30130,0.0553,0.029233,0.000000,0.000000,0.017540
1,scaleway,fr-par,46.235196,0.31280,0.0553,0.029233,0.045872,0.000000,0.029008
2,ovh,bhs,29.925137,0.30392,0.0638,0.007656,0.010451,0.171026,0.032860
3,hetzner,hel1,59.544060,0.30130,0.0595,0.046840,0.000000,0.084507,0.040780
4,ovh,bhs,29.925137,0.31552,0.0638,0.007656,0.056721,0.171026,0.044428
5,ovh,rbx,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
6,ovh,sbg,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
7,ovh,gra,46.235196,0.30392,0.0638,0.029233,0.010451,0.171026,0.045807
8,hetzner,hel1,59.544060,0.31280,0.0595,0.046840,0.045872,0.084507,0.052248
9,ovh,gra,46.235196,0.31552,0.0638,0.029233,0.056721,0.171026,0.057374


In [28]:
results_dir = BASE_DIR / "results"
results_dir.mkdir(exist_ok=True)

ranked_candidates.to_csv(
    results_dir / "ai_scheduler_ranked_regions.csv",
    index=False
)

print("AI scheduler ranking saved.")

AI scheduler ranking saved.


In [29]:
workload_tasks = processed_df.head(10000).copy().reset_index(drop=True)

resource_pool = scheduler_resources.copy().reset_index(drop=True)

resource_pool = resource_pool.sort_values(
    ["predicted_carbon_intensity", "cost_per_hour"]
).reset_index(drop=True)

resource_pool["available_cpu"] = resource_pool["total_cpu_mips"]
resource_pool["available_memory"] = resource_pool["ram_mb"]

print("Workloads:", len(workload_tasks))
print("Resources:", len(resource_pool))

display(workload_tasks.head())
display(
    resource_pool[
        ["provider", "region", "host_id", "available_cpu", "available_memory"]
    ].head()
)

Workloads: 10000
Resources: 1160


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


,provider,region,host_id,available_cpu,available_memory
0,gcp,europe-west6,H06,3200,8192
1,gcp,europe-west6,H01,3200,8192
2,azure,switzerlandnorth,H06,3200,8192
3,azure,switzerlandnorth,H01,3200,8192
4,gcp,europe-west6,H07,8000,16384


In [30]:
workload_tasks = processed_df.head(10000).copy().reset_index(drop=True)

resource_pool = scheduler_resources.copy().reset_index(drop=True)

resource_pool = resource_pool.sort_values(
    ["predicted_carbon_intensity", "cost_per_hour"]
).reset_index(drop=True)

resource_pool["available_cpu"] = resource_pool["total_cpu_mips"]
resource_pool["available_memory"] = resource_pool["ram_mb"]

print("Workloads:", len(workload_tasks))
print("Resources:", len(resource_pool))

display(workload_tasks.head())
display(
    resource_pool[
        ["provider", "region", "host_id", "available_cpu", "available_memory"]
    ].head()
)

Workloads: 10000
Resources: 1160


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


,provider,region,host_id,available_cpu,available_memory
0,gcp,europe-west6,H06,3200,8192
1,gcp,europe-west6,H01,3200,8192
2,azure,switzerlandnorth,H06,3200,8192
3,azure,switzerlandnorth,H01,3200,8192
4,gcp,europe-west6,H07,8000,16384


In [2]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

while not (BASE_DIR / "data" / "processed").exists():
    if BASE_DIR.parent == BASE_DIR:
        raise FileNotFoundError("Project folder not found")
    BASE_DIR = BASE_DIR.parent

WORKLOAD_PATH = BASE_DIR / "data" / "processed" / "workload.csv"

workload_df = pd.read_csv(WORKLOAD_PATH, nrows=10000)

print("Workload loaded:", workload_df.shape)

Workload loaded: (10000, 5)


In [3]:
print(workload_df.head(10))
print(workload_df[["CPU", "Memory", "Arrival_Time"]].describe())

         Job_ID  Task_ID  Arrival_Time       CPU    Memory
0  736958118313        0    601263.722  0.017989  0.000057
1  736958118321        0    601339.991  0.008105  0.000057
2  736958118528        0    601975.491  0.112489  0.000237
3  736958125656        0    604299.116  0.055855  0.001207
4  736959135511        0    608551.032  0.323642  0.003185
5  736959135802        0    609764.644  0.289879  0.003292
6  736959135804        0    609781.475  0.324837  0.003220
7  736959135806        0    609781.502  0.275387  0.003002
8  736959135808        0    609781.529  0.342675  0.003368
9  736959135810        0    609781.556  0.321858  0.003277
                CPU        Memory  Arrival_Time
count  10000.000000  10000.000000  1.000000e+04
mean       4.334530      0.002004  2.775532e+06
std       14.169670      0.002612  1.318874e+06
min        0.000005      0.000003  6.012637e+05
25%        0.019209      0.000550  1.586838e+06
50%        0.154187      0.001114  2.864908e+06
75%        3.57

In [4]:
raw_path = BASE_DIR / "data" / "raw" / "Cell_c.csv"

raw_df = pd.read_csv(raw_path, nrows=10)

print(raw_df.shape)
print(raw_df.columns.tolist())
display(raw_df.head())

(10, 5)
['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']


,Job_ID,Task_ID,Arrival_Time,CPU,Memory
0,736958118313,0,601263.722,0.017989,0.000057
1,736958118321,0,601339.991,0.008105,0.000057
2,736958118528,0,601975.491,0.112489,0.000237
3,736958125656,0,604299.116,0.055855,0.001207
4,736959135511,0,608551.032,0.323642,0.003185


In [5]:
history_path = BASE_DIR / "data" / "raw" / "history.csv"

history_df = pd.read_csv(history_path, nrows=10)

print("Shape:", history_df.shape)
print("Columns:", history_df.columns.tolist())
display(history_df.head(10))

Shape: (10, 5)
Columns: ['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage']


,provider,region,timestamp,carbon_intensity_gco2_kwh,renewable_percentage
0,aws,af-south-1,2026-08-19T00:53:00.211592+00:00,795.6,5.0
1,aws,af-south-1,2026-08-19T02:17:53.174894+00:00,795.6,5.0
2,aws,af-south-1,2026-08-19T03:42:26.812833+00:00,780.0,8.0
3,aws,af-south-1,2026-08-19T04:34:02.547742+00:00,780.0,8.0
4,aws,af-south-1,2026-08-19T05:25:46.493498+00:00,780.0,8.0
5,aws,af-south-1,2026-08-19T06:36:13.159117+00:00,780.0,8.0
6,aws,af-south-1,2026-08-19T07:38:12.934391+00:00,780.0,8.0
7,aws,af-south-1,2026-08-19T08:35:11.965762+00:00,717.6,14.0
8,aws,af-south-1,2026-08-19T09:29:21.138909+00:00,717.6,14.0
9,aws,af-south-1,2026-08-19T10:31:30.130076+00:00,717.6,14.0


In [6]:
print("Workload file:", raw_path)
print("Workload columns:", raw_df.columns.tolist())
print("History columns:", history_df.columns.tolist())

Workload file: c:\Users\LENOVO\Downloads\AI-Carbon-Aware-Cloud-Scheduler\AI-Carbon-Aware-Cloud-Scheduler\data\raw\Cell_c.csv
Workload columns: ['Job_ID', 'Task_ID', 'Arrival_Time', 'CPU', 'Memory']
History columns: ['provider', 'region', 'timestamp', 'carbon_intensity_gco2_kwh', 'renewable_percentage']


In [7]:
region_path = BASE_DIR / "data" / "processed" / "region_config.csv"

region_df = pd.read_csv(region_path)

display(
    region_df[
        [
            "provider",
            "region",
            "pes",
            "mips_per_pe",
            "total_cpu_mips",
            "ram_mb",
            "idle_power_w",
            "max_power_w",
            "pue",
            "cost_per_hour"
        ]
    ].describe()
)

,pes,mips_per_pe,total_cpu_mips,ram_mb,idle_power_w,max_power_w,pue,cost_per_hour
count,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000,1160.000000
mean,9.600000,1300.000000,14160.000000,19660.800000,178.200000,355.000000,1.174310,0.086405
std,4.080975,419.704485,10122.387561,8357.836864,31.717933,64.160336,0.016783,0.010758
min,4.000000,800.000000,3200.000000,8192.000000,130.000000,262.000000,1.150000,0.055300
25%,8.000000,1000.000000,8000.000000,16384.000000,160.000000,324.000000,1.160000,0.080800
50%,8.000000,1200.000000,9600.000000,16384.000000,171.000000,335.000000,1.180000,0.085500
75%,12.000000,1500.000000,18000.000000,24576.000000,202.000000,398.000000,1.180000,0.095000
max,16.000000,2000.000000,32000.000000,32768.000000,232.000000,460.000000,1.200000,0.105000
